# Notebook 4 — Advanced Python Patterns

**Topics covered:** First-class functions & closures · Decorators · Iterators · Generators · Type Annotations

| Topic | Subtopics |
|---|---|
| 1 · First-class functions & closures | Functions as values, closures, returning functions |
| 2 · Decorators | Basic wrappers, `functools.wraps`, stacking, decorators with arguments |
| 3 · Iterators | `__iter__` / `__next__`, built-in `iter()` / `next()` |
| 4 · Generators | `yield`, generator expressions, `send()` |
| 5 · Type Annotations | Variable & function annotations, `typing` module |

---

### Sample Data
All examples use two consistent person records — run the setup cell below before any other cell.

In [ ]:
# Shared sample data used throughout this notebook
people = [
    {
        "first_name": "Hanu",
        "last_name":  "Madala",
        "dob":        "20 Jan 2002",
        "phone":      "+65 1234 5678",
        "email":      "hanu.m@tdfs.com",
    },
    {
        "first_name": "Shankar",
        "last_name":  "Cherukuri",
        "dob":        "30 Mar 1997",
        "phone":      "98765432",
        "email":      "shankar.c@tdfs.com",
    },
]

---
## Topic 1 — First-class Functions & Closures

In Python, functions are first-class objects: they can be stored in variables, passed as arguments, and returned from other functions. Closures capture variables from the enclosing scope, enabling stateful function factories.

#### 1a · Functions as first-class objects — assigning, passing, storing

A function can be assigned to a variable or passed to another function just like any other value, without calling it.

In [ ]:
def full_name(person):
    return f"{person['first_name']} {person['last_name']}"

def format_email(person):
    return person["email"].lower()

# Store functions in a list — call them uniformly
formatters = [full_name, format_email]

for person in people:
    row = [fn(person) for fn in formatters]   # fn is a function object
    print(" | ".join(row))

#### 1b · Closures — inner functions capturing enclosing-scope variables

A closure is an inner function that remembers a variable from its enclosing function's scope even after that outer function has returned.

In [ ]:
def make_greeter(prefix):
    """Returns a function that greets a person with the given prefix."""
    def greet(person):
        # 'prefix' is captured from the enclosing make_greeter scope
        return f"{prefix} {person['first_name']} {person['last_name']}!"
    return greet

formal   = make_greeter("Good morning,")
informal = make_greeter("Hey,")

for person in people:
    print(formal(person))
    print(informal(person))

#### 1c · Stateful closures — mutable state via `nonlocal`

`nonlocal` lets an inner function rebind (not just read) a variable in the enclosing scope, enabling simple stateful factories without a class.

In [ ]:
def make_call_counter(func):
    """Wraps func and tracks how many times it has been called."""
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        result = func(*args, **kwargs)
        print(f"  [{func.__name__} called {count}x] → {result}")
        return result
    wrapper.call_count = lambda: count   # expose counter
    return wrapper

tracked_full_name = make_call_counter(full_name)

for person in people:
    tracked_full_name(person)

print(f"\nTotal calls: {tracked_full_name.call_count()}")

---
## Topic 2 — Decorators

A decorator is applied with the `@name` syntax directly above a function or class definition. Python ships with several powerful built-in and standard-library decorators that you can use without knowing how to build one.

#### 2a · Single decorator — `@property`

`@property` turns a method into a read-only computed attribute accessed without parentheses — one of the most common built-in decorators.

In [ ]:
class Person:
    def __init__(self, first_name, last_name, dob, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.email      = email

    @property
    def full_name(self):
        # @property: access as hanu.full_name — no () needed
        return f"{self.first_name} {self.last_name}"

    @property
    def email_domain(self):
        return self.email.split("@")[-1]

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "shankar.c@tdfs.com")

print(hanu.full_name)           # attribute access — no ()
print(shankar.full_name)
print(hanu.email_domain)        # tdfs.com
print(shankar.email_domain)     # tdfs.com

#### 2b · Single decorator — `@functools.lru_cache`

`@functools.lru_cache` caches return values by argument; subsequent calls with the same argument return the stored result instantly without re-running the function body.

In [ ]:
import functools
from datetime import datetime

@functools.lru_cache(maxsize=128)
def calculate_age(dob_str: str) -> int:
    """Parse dob_str and return age in full years. Result is cached per dob_str."""
    dob   = datetime.strptime(dob_str, "%d %b %Y")
    today = datetime.today()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

# First call — result computed and placed in the cache
for person in people:
    age = calculate_age(person["dob"])
    print(f"{person['first_name']}: {age} years old")

# Second call with the same argument — served from cache, no re-computation
print("\nCalling again with same DOB (from cache):")
print(calculate_age("20 Jan 2002"))

# Inspect how many times the cache was hit vs missed
print("\nCache info:", calculate_age.cache_info())

#### 2c · Stacking decorators — applying multiple decorators to one function

Multiple decorators are applied bottom-up: `@A` on top of `@B` means `A(B(func))`. The outermost decorator runs first when the function is called.

In [ ]:
import functools

def log_call(func):
    @functools.wraps(func)          # copies __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        print(f"[LOG] calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

def timeit(func):
    import time
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0     = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        print(f"[TIMEIT] {func.__name__} took {elapsed_ms:.3f} ms")
        return result
    return wrapper

@log_call          # applied second (outer) — runs first on call
@timeit            # applied first (inner) — runs second on call
def process_person(person):
    """Returns a formatted summary string for a person dict."""
    age = calculate_age(person["dob"])
    return f"{person['first_name']} {person['last_name']}, age {age}"

print("--- Hanu ---")
print(process_person(people[0]))
print()
print("--- Shankar ---")
print(process_person(people[1]))

---
## Topic 3 — Iterators

An iterator is an object that produces values one at a time via `__next__()`. The iterator protocol requires both `__iter__()` (returns self) and `__next__()` (returns next value or raises `StopIteration`).

#### 3a · Custom iterator class — `__iter__`, `__next__`, `StopIteration`

Implement `__iter__` to return the iterator object itself and `__next__` to advance and return the next value; raise `StopIteration` when the sequence is exhausted.

In [ ]:
class PersonRoster:
    """Iterates over a list of person dicts, yielding one at a time."""

    def __init__(self, persons):
        self._persons = persons
        self._index   = 0

    def __iter__(self):
        return self          # the iterator IS the iterable

    def __next__(self):
        if self._index >= len(self._persons):
            raise StopIteration
        person = self._persons[self._index]
        self._index += 1
        return f"{self._index}. {person['first_name']} {person['last_name']} ({person['email']})"

roster = PersonRoster(people)

for entry in roster:
    print(entry)

---
## Topic 4 — Generators

A generator function uses `yield` to produce values lazily — one at a time — without building the entire sequence in memory. Python automatically creates the iterator object when the generator function is called.

#### 4a · Generator functions — `yield`, lazy evaluation

Each time `yield` is reached, the function suspends and returns the yielded value; execution resumes from that point on the next `next()` call.

In [ ]:
def filter_by_domain(persons, domain):
    """Lazily yields persons whose email ends with the given domain."""
    for person in persons:
        if person["email"].endswith(domain):
            print(f"email: {person['email']} ...")   # runs only when next() is called
            yield person           # suspend here, resume on next call

gen = filter_by_domain(people, "tdfs.com")

print(type(gen))   # <class 'generator'>
print("--- consuming generator ---")

for person in gen:
    print(f"Person details: {person['first_name']} {person['last_name']} | {person['email']}")


#### 4b · Generator expressions — inline lazy sequences

A generator expression uses parentheses `(expr for item in iterable [if condition])` and returns a generator object without materialising the sequence, unlike a list comprehension.

In [ ]:
# List comprehension — builds the full list in memory immediately
full_names_list = [f"{p['first_name']} {p['last_name']}" for p in people]

# Generator expression — produces values lazily, one at a time
full_names_gen = (f"{p['first_name']} {p['last_name']}" for p in people)

print("List:", full_names_list)    # already evaluated
print("Generator:", full_names_gen)  # not yet evaluated

print("\nConsuming generator:")
for name in full_names_gen:
    print(f"  {name}")

# Generator expressions work directly inside function calls
emails = ", ".join(p["email"] for p in people)
print(f"\nEmails: {emails}")

---
## Topic 5 — Type Annotations

Type annotations document the expected types of variables and function signatures. They are not enforced at runtime by default but are used by static type checkers (e.g. mypy, Pylance) and improve IDE support and readability.

#### 5a · Variable and function annotations — `:`, `->` syntax

Annotate variables with `name: type = value` and function parameters/return types with `param: type` and `-> return_type`; annotations are stored in `__annotations__` but not checked at runtime.

In [ ]:
from datetime import date, datetime

# Variable annotations
first_name: str = "Hanu"
dob_year:   int = 2002
is_active:  bool = True

# Function annotations
def full_name(first: str, last: str) -> str:
    return f"{first} {last}"

def calculate_age(dob_str: str, today: date | None = None) -> int:
    """Returns the person's age in full years from a 'DD Mon YYYY' string."""
    dob_date = datetime.strptime(dob_str, "%d %b %Y").date()
    today    = today or date.today()
    return today.year - dob_date.year - (
        (today.month, today.day) < (dob_date.month, dob_date.day)
    )

# __annotations__ shows declared types
print(calculate_age.__annotations__)

for person in people:
    name = full_name(person["first_name"], person["last_name"])
    age  = calculate_age(person["dob"])
    print(f"  {name}, age {age}")

---
## Exercises

Complete each exercise in the cell below it. Reference solutions are hidden in the `<details>` blocks.

### Exercise 1 — `@property` on a Person class

Create a `Person` class with `__init__` taking `first_name`, `last_name`, `dob`, and `email`. Add two `@property` methods:
- `full_name` → `"First Last"`
- `email_domain` → the part after `@` in the email

Instantiate both Hanu and Shankar and print their `full_name` and `email_domain`.

<details>
<summary>Solution</summary>

```python
class Person:
    def __init__(self, first_name, last_name, dob, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.email      = email

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

    @property
    def email_domain(self):
        return self.email.split("@")[-1]

for p in people:
    person = Person(p["first_name"], p["last_name"], p["dob"], p["email"])
    print(person.full_name, "|", person.email_domain)
```

</details>

In [ ]:
# Exercise 1 — your code here

### Exercise 2 — `@dataclass`

Use `@dataclass` to define a `Person` class with fields `first_name`, `last_name`, `dob`, and `email`. Add a `full_name` property.

Instantiate both Hanu and Shankar using the `people` list and print their `repr()` and `full_name`. Verify that `@dataclass` generated `__repr__` automatically.

<details>
<summary>Solution</summary>

```python
from dataclasses import dataclass

@dataclass
class Person:
    first_name: str
    last_name:  str
    dob:        str
    email:      str

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

for p in people:
    person = Person(p["first_name"], p["last_name"], p["dob"], p["email"])
    print(repr(person))
    print(person.full_name)
```

</details>

In [ ]:
# Exercise 2 — your code here

### Exercise 3 — `PersonRoster` iterator with type hints

Build a `PersonRoster` iterator class initialised with a list of person dicts. It should yield one person per iteration as a formatted string `"N. First Last"`. Annotate all method signatures with proper type hints.

<details>
<summary>Solution</summary>

```python
from typing import List, Dict, Iterator

class PersonRoster:
    def __init__(self, persons: List[Dict[str, str]]) -> None:
        self._persons: List[Dict[str, str]] = persons
        self._index: int = 0

    def __iter__(self) -> Iterator[str]:
        return self

    def __next__(self) -> str:
        if self._index >= len(self._persons):
            raise StopIteration
        p = self._persons[self._index]
        self._index += 1
        return f"{self._index}. {p['first_name']} {p['last_name']}"

for entry in PersonRoster(people):
    print(entry)
```

</details>

In [ ]:
# Exercise 3 — your code here

### Exercise 4 — `filter_by_domain` generator

Write a generator function `filter_by_domain(persons, domain)` that lazily yields only persons whose email ends with the given domain. Use it to filter for `"tdfs.com"` and print `"First Last — email"` for each match.

<details>
<summary>Solution</summary>

```python
def filter_by_domain(persons, domain):
    for p in persons:
        if p["email"].endswith(domain):
            yield p

for p in filter_by_domain(people, "tdfs.com"):
    print(f"{p['first_name']} {p['last_name']} — {p['email']}")
```

</details>

In [ ]:
# Exercise 4 — your code here

### Exercise 5 — `@functools.lru_cache`

Apply `@functools.lru_cache(maxsize=128)` to a `calculate_age(dob_str)` function. Call it twice for each person's DOB and then print `cache_info()` to confirm that the second calls were served from the cache (check `hits`).

<details>
<summary>Solution</summary>

```python
import functools
from datetime import datetime

@functools.lru_cache(maxsize=128)
def calculate_age(dob_str: str) -> int:
    dob   = datetime.strptime(dob_str, "%d %b %Y")
    today = datetime.today()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

# First pass — cache misses
for p in people:
    print(f"{p['first_name']}: {calculate_age(p['dob'])}")

# Second pass — cache hits
for p in people:
    calculate_age(p["dob"])

print(calculate_age.cache_info())   # hits=2, misses=2
```

</details>

In [ ]:
# Exercise 5 — your code here

In [ ]:
# Exercise 6 — your code here

### Exercise 6 — Fully type-annotated `summarise()`

Write a fully type-annotated function `summarise(persons: List[Dict[str, str]]) -> List[str]` that returns strings in the format `"Hanu Madala, age 24"`. Run it against the `people` list and verify the output.

<details>
<summary>Solution</summary>

```python
from typing import List, Dict
from datetime import datetime

def summarise(persons: List[Dict[str, str]]) -> List[str]:
    def age(dob_str: str) -> int:
        dob   = datetime.strptime(dob_str, "%d %b %Y")
        today = datetime.today()
        return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

    return [
        f"{p['first_name']} {p['last_name']}, age {age(p['dob'])}"
        for p in persons
    ]

for summary in summarise(people):
    print(summary)
```

</details>

In [ ]:
# Exercise 7 — your code here